# E207 Final: Fast Music Retrieval Under Time Warping

Collaborators: Jordan Carlin, Jasper Cox, Thomas Lilygren

Notes from Tsai:
Coarse quantization - too low move up higher iteratively
Block feature space into 
Template with highest activation energy - most robust

In [1]:
%matplotlib inline

In [2]:
import numpy as np
import librosa as lb
import matplotlib.pyplot as plt
import IPython.display as ipd
import scipy.io as sio
import scipy.signal
import glob
import os.path
import subprocess
import pickle
import difflib
import string
from pathlib import Path
from typing import Literal
from sklearn.decomposition import non_negative_factorization as NMF

In [13]:
# Configuration
N_FFT = 2048
HOP_LENGTH = 512
EPS = np.finfo(np.float64).eps  # small constant to avoid division by zero
RANK = 2
LETTERS = list(string.ascii_lowercase)[0:RANK]

In [4]:
def load_audio(audio_file: Path) -> tuple[np.ndarray, int | float]:
    """
    Load an audio file. Returns a tuple of the audio time series and the sample rate.
    """
    audio, sr = lb.load(audio_file, sr=None)
    return audio, sr

In [5]:
def time_to_freq(audio: np.ndarray, sr: int | float, algorithim: Literal["stft", "cqt", "logmel", "chroma"] = "stft", n_fft: int = N_FFT, hop_length: int = HOP_LENGTH) -> np.ndarray:
    """
    Convert a time-domain audio signal to a frequency-domain representation.
    """
    match algorithim:
        case "stft":
            X = lb.stft(audio, n_fft=n_fft, hop_length=hop_length)
            return X
        case "cqt":
            C = lb.cqt(audio, sr=sr, hop_length=hop_length)
            return C
        case "logmel":
            S = lb.feature.melspectrogram(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=128)
            log_S = lb.power_to_db(S, ref=np.max)
            return log_S
        case "chroma":
            C = lb.feature.chroma_stft(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length)
            return C
        case _:
            raise ValueError(f"Unsupported type: {algorithim}")

In [25]:
def extract_highest_energy(H: np.ndarray, median_frames: int = 7) -> np.ndarray:
    """For each time frame, return the index of the highest-activation component.

    Applies median filtering across time to smooth out frame-level noise.
    Frames where the peak activation is below `threshold` * global max are masked as -1 (silence).
    """
    # Smooth activations along time axis to reduce spurious jumps
    H_smooth = scipy.signal.medfilt2d(H.astype(np.float64), kernel_size=(1, median_frames))
    dominant = np.argmax(H_smooth, axis=0)
    #peak_per_frame = H_smooth[dominant, np.arange(H_smooth.shape[1])]
    # silence_mask = peak_per_frame < threshold * peak_per_frame.max()
    # dominant[silence_mask] = -1
    return dominant

In [35]:
def test_single_file(audio_file: Path, n_components=RANK) -> None:
    audio, sr = load_audio(audio_file)
    X = time_to_freq(audio, sr, "stft")
    X = np.abs(X)

    W, H, n_iter = NMF(X, init='random', solver='mu', n_components=n_components)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    im0 = axes[0].imshow(W, aspect="auto", origin="lower", interpolation="none")
    axes[0].set_title("W (Basis Vectors / MIDI Templates)")
    #axes[0].set_xlabel("MIDI Note (21=A0 ... 108=C8)")
    axes[0].set_ylabel("Frequency Bin")
    fig.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(H, aspect="auto", origin="lower", interpolation="none")
    axes[1].set_title("H (Piano Roll)")
    axes[1].set_xlabel("Time Frame")
    #axes[1].set_ylabel("MIDI Note Index")
    fig.colorbar(im1, ax=axes[1])

    plt.tight_layout()
    plt.show()

    dominant_component = extract_highest_energy(H)
    #active = dominant_component >= 0
    dominant_component = np.array(LETTERS)[dominant_component]#[active]]
    n_frames = np.arange(H.shape[1])
    #times = lb.frames_to_time(np.where(active)[0], sr=sr, hop_length=HOP_LENGTH)
    plt.figure(figsize=(12, 4))
    plt.scatter(n_frames, dominant_component, s=5)
    plt.xlabel("Frames")
    #plt.xlim([0, 30])
    #plt.ylabel("MIDI Note")
    plt.title("Dominant Note Over Time")
    #plt.yticks(range(MIDI_MIN, MIDI_MAX + 1, 12), [lb.midi_to_note(m) for m in range(MIDI_MIN, MIDI_MAX + 1, 12)])
    plt.show()

In [ ]:
print("File 1")
test_single_file(Path("./songs/Sonata in C Minor, D958/MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.wav"))

File 1


In [ ]:
print("File 2")
test_single_file(Path("./songs/Sonata in C Minor, D958/MIDI-Unprocessed_Schubert4-6_MID--AUDIO_08_R2_2018_wav.wav"))